In [1]:
# Spark Session to read and print employees_records.csv
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeRecordsSession") \
    .getOrCreate()

# sc = spark.sparkContext

orders = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
orders.show(10)
# Create 3 temporary tables/views
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")

+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|order_id|customer_id|product_id|order_date| ship_date|quantity|unit_price|discount_percentage|payment_mode|order_status|      sales_amount|
+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|       1|         45|        27|2025-06-16|2025-06-22|       4|      6479|                 15|  Debit Card|   Delivered|           22028.6|
|       2|         64|        19|2026-02-14|2026-02-21|       3|     27477|                 22|         UPI|     Shipped|          64296.18|
|       3|         15|        50|2026-03-23|2026-03-30|       3|     17041|                 10|         UPI|     Shipped|46010.700000000004|
|       4|         47|        40|2025-10-08|2025-10-16|       1|     12793|                 16| Net Banking|   Delivered|10746.119999999999|
|       5|   

In [16]:
spark.sql("""
select c.customer_id , c.customer_name, sum(o.sales_amount) as total_rev
from customers c join orders o on c.customer_id=o.customer_id 
group by c.customer_id,c.customer_name
order 
by total_rev desc limit 10
""").show()

+-----------+----------------+-----------------+
|customer_id|   customer_name|        total_rev|
+-----------+----------------+-----------------+
|         93|Shirley Harrison|        845556.64|
|          6|Theresa Ferguson|        720304.25|
|         11|   Darrell Jones|592001.0900000001|
|         45|    Jason Nelson|        591651.49|
|         80|    Jason Garcia|        588679.86|
|        106|    Shane Miller|         581160.2|
|         52| Jennifer Hunter|        565407.64|
|         56|   Matthew Evans|        532683.28|
|         64|     Nicole Meza|        503299.37|
|        104| Alison Mcdonald|        497268.72|
+-----------+----------------+-----------------+



In [ ]:
#Monthly Revenue Trend 
spark.sql("""
select date_Format(order_date,"yyyy-MM") as order_month, sum(sales_amount) as monthly_revenue
from orders
group by order_month
ORDER BY order_month
""").show()
#'yyyy-MM-dd HH:mm:ss.SSS'

+-----------+------------------+
|order_month|   monthly_revenue|
+-----------+------------------+
|    2025-05|2780923.4999999986|
|    2025-06|3047238.7300000004|
|    2025-07|2519717.0100000002|
|    2025-08|3116055.6300000004|
|    2025-09|2073947.1400000001|
|    2025-10|2432460.0199999996|
|    2025-11|3146286.3499999987|
|    2025-12|        2698370.84|
|    2026-01|2728039.2299999995|
|    2026-02|        1875279.53|
|    2026-03| 3167027.819999999|
|    2026-04|2734291.4299999997|
|    2026-05|         738030.75|
+-----------+------------------+



In [ ]:
#cummulative revenue
spark.sql("""
select order_id,order_date,sales_amount,
sum(sales_amount) over(
   order by order_id, order_date
   rows between unbounded preceding and current row


)as row_cum
from orders 
order by order_date, order_id


""").show() #ROWS BETWEEN start AND end <- syntax

+--------+----------+------------------+--------------------+
|order_id|order_date|      sales_amount|             row_cum|
+--------+----------+------------------+--------------------+
|     122|2025-05-07|           18759.0|   7059144.919999997|
|     225|2025-05-07|           5405.92|       1.495544712E7|
|     328|2025-05-07|          79520.64|2.1105373810000006E7|
|     385|2025-05-07|          75493.48| 2.426880826999999E7|
|      94|2025-05-08|           39448.5|   5667693.999999999|
|     318|2025-05-08|          41910.88|2.0489456140000004E7|
|      69|2025-05-09| 75555.23999999999|          4198953.21|
|      81|2025-05-10|          98437.92|   4889629.829999999|
|     150|2025-05-10|          85624.75|   9268375.349999996|
|     238|2025-05-10|           43851.0|       1.575491874E7|
|     339|2025-05-10|          165430.8|       2.171266492E7|
|     449|2025-05-10|          97460.28|2.9520937779999986E7|
|     252|2025-05-12|          37186.89|1.6565951140000002E7|
|     31

In [29]:
#previous order analysis - lag
spark.sql("""
select 
customer_id,order_id,order_date,sales_amount,
LAG(sales_amount) over(
partition by customer_id order by order_date
) as previous_purchase
from orders
ORDER BY customer_id, order_date
""").show()

+-----------+--------+----------+------------------+------------------+
|customer_id|order_id|order_date|      sales_amount| previous_purchase|
+-----------+--------+----------+------------------+------------------+
|          1|     251|2025-09-19|          126064.4|              NULL|
|          1|     424|2025-10-08|          30719.96|          126064.4|
|          2|     443|2025-10-06|         108951.48|              NULL|
|          2|     306|2025-11-21|          31602.42|         108951.48|
|          2|     174|2025-12-04|196283.40000000002|          31602.42|
|          2|     378|2026-03-01|           59844.0|196283.40000000002|
|          3|     402|2025-06-03|          32065.68|              NULL|
|          3|     347|2025-09-18|2446.9900000000002|          32065.68|
|          3|     400|2026-01-13|22083.600000000002|2446.9900000000002|
|          3|     101|2026-03-12|         131495.24|22083.600000000002|
|          4|     349|2025-07-07|125330.40000000001|            

In [6]:
#Previous Order Analysis — LAG
spark.sql("""
SELECT
    customer_id,
    order_id,
    order_date,
    sales_amount,

    LAG(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_purchase

FROM orders
ORDER BY customer_id, order_date
""").show()

+-----------+--------+----------+------------------+------------------+
|customer_id|order_id|order_date|      sales_amount| previous_purchase|
+-----------+--------+----------+------------------+------------------+
|          1|     251|2025-09-19|          126064.4|              NULL|
|          1|     424|2025-10-08|          30719.96|          126064.4|
|          2|     443|2025-10-06|         108951.48|              NULL|
|          2|     306|2025-11-21|          31602.42|         108951.48|
|          2|     174|2025-12-04|196283.40000000002|          31602.42|
|          2|     378|2026-03-01|           59844.0|196283.40000000002|
|          3|     402|2025-06-03|          32065.68|              NULL|
|          3|     347|2025-09-18|2446.9900000000002|          32065.68|
|          3|     400|2026-01-13|22083.600000000002|2446.9900000000002|
|          3|     101|2026-03-12|         131495.24|22083.600000000002|
|          4|     349|2025-07-07|125330.40000000001|            

In [7]:
#add difference between current and previous purchase
spark.sql("""
SELECT
    customer_id,
    order_id,
    order_date,
    sales_amount,

    LAG(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_purchase,

    sales_amount -
    LAG(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS purchase_difference

FROM orders
ORDER BY customer_id, order_date
""").show()

+-----------+--------+----------+------------------+------------------+-------------------+
|customer_id|order_id|order_date|      sales_amount| previous_purchase|purchase_difference|
+-----------+--------+----------+------------------+------------------+-------------------+
|          1|     251|2025-09-19|          126064.4|              NULL|               NULL|
|          1|     424|2025-10-08|          30719.96|          126064.4|          -95344.44|
|          2|     443|2025-10-06|         108951.48|              NULL|               NULL|
|          2|     306|2025-11-21|          31602.42|         108951.48|          -77349.06|
|          2|     174|2025-12-04|196283.40000000002|          31602.42| 164680.98000000004|
|          2|     378|2026-03-01|           59844.0|196283.40000000002|-136439.40000000002|
|          3|     402|2025-06-03|          32065.68|              NULL|               NULL|
|          3|     347|2025-09-18|2446.9900000000002|          32065.68|         

In [8]:
#Next Purchase Prediction — LEAD
spark.sql("""
SELECT
    customer_id,
    order_id,
    order_date,
    sales_amount,

    LEAD(sales_amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS next_purchase_amount

FROM orders
ORDER BY customer_id, order_date
""").show()

+-----------+--------+----------+------------------+--------------------+
|customer_id|order_id|order_date|      sales_amount|next_purchase_amount|
+-----------+--------+----------+------------------+--------------------+
|          1|     251|2025-09-19|          126064.4|            30719.96|
|          1|     424|2025-10-08|          30719.96|                NULL|
|          2|     443|2025-10-06|         108951.48|            31602.42|
|          2|     306|2025-11-21|          31602.42|  196283.40000000002|
|          2|     174|2025-12-04|196283.40000000002|             59844.0|
|          2|     378|2026-03-01|           59844.0|                NULL|
|          3|     402|2025-06-03|          32065.68|  2446.9900000000002|
|          3|     347|2025-09-18|2446.9900000000002|  22083.600000000002|
|          3|     400|2026-01-13|22083.600000000002|           131495.24|
|          3|     101|2026-03-12|         131495.24|                NULL|
|          4|     349|2025-07-07|12533

In [9]:
# Customer Retention — Consecutive Months
spark.sql("""
WITH monthly_orders AS (

    SELECT DISTINCT
        customer_id,
        DATE_FORMAT(order_date, 'yyyy-MM') AS order_month
    FROM orders

),

previous_month AS (

    SELECT
        customer_id,
        order_month,

        LAG(order_month) OVER (
            PARTITION BY customer_id
            ORDER BY order_month
        ) AS prev_month

    FROM monthly_orders
)

SELECT
    customer_id,
    order_month,
    prev_month

FROM previous_month

WHERE
    MONTHS_BETWEEN(
        TO_DATE(CONCAT(order_month, '-01')),
        TO_DATE(CONCAT(prev_month, '-01'))
    ) = 1

ORDER BY customer_id, order_month
""").show()

+-----------+-----------+----------+
|customer_id|order_month|prev_month|
+-----------+-----------+----------+
|          1|    2025-10|   2025-09|
|          2|    2025-11|   2025-10|
|          2|    2025-12|   2025-11|
|          6|    2025-06|   2025-05|
|          6|    2025-07|   2025-06|
|          6|    2025-08|   2025-07|
|          8|    2026-03|   2026-02|
|         11|    2026-03|   2026-02|
|         13|    2026-01|   2025-12|
|         13|    2026-02|   2026-01|
|         14|    2026-04|   2026-03|
|         15|    2025-12|   2025-11|
|         15|    2026-03|   2026-02|
|         16|    2025-11|   2025-10|
|         18|    2025-06|   2025-05|
|         19|    2025-08|   2025-07|
|         20|    2025-06|   2025-05|
|         20|    2025-12|   2025-11|
|         24|    2025-06|   2025-05|
|         24|    2026-03|   2026-02|
+-----------+-----------+----------+
only showing top 20 rows



In [10]:
#7. Highest Selling Product Per Category
spark.sql("""
WITH product_sales AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(o.quantity) AS total_quantity

    FROM products p
    JOIN orders o
        ON p.product_id = o.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (

    SELECT
        *,
        RANK() OVER (
            PARTITION BY category
            ORDER BY total_quantity DESC
        ) AS product_rank

    FROM product_sales
)

SELECT
    category,
    product_id,
    product_name,
    total_quantity

FROM ranked_products

WHERE product_rank = 1

ORDER BY category
""").show()

+-----------+----------+--------------+--------------+
|   category|product_id|  product_name|total_quantity|
+-----------+----------+--------------+--------------+
|Electronics|        44|Worker Product|            41|
|    Fashion|        43| Build Product|            42|
|  Furniture|         7| Place Product|            56|
|    Grocery|        23|   Say Product|            32|
|    Grocery|        19|  Fill Product|            32|
|    Grocery|        28|   Car Product|            32|
|     Sports|        27| Visit Product|            38|
+-----------+----------+--------------+--------------+



In [11]:
#8. Average Shipping Delay
spark.sql("""
SELECT
    AVG(DATEDIFF(ship_date, order_date)) AS average_shipping_delay
FROM orders
WHERE ship_date IS NOT NULL
""").show()

+----------------------+
|average_shipping_delay|
+----------------------+
|                 5.672|
+----------------------+

